In [ ]:
from transformer_lens import HookedTransformer
import torch
import plotly.express as px
import pandas as pd

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

model: HookedTransformer = HookedTransformer.from_pretrained("pythia-14M")

In [ ]:
logits, cache = model.run_with_cache("The sky is blue and the grass is green.")
prompt = "The sky is blue and the grass is green."
focus_token_idx = -1
print(logits.shape)
#print(cache.keys())

for k,v in cache.items():
    print(k, v.shape)

In [ ]:
tokens = model.to_str_tokens(prompt)
n_layers = model.cfg.n_layers

In [ ]:
def logit_lens(cache, model):
    """Returns (n_layers, seq_len, vocab_size) tensor of logits at each layer."""
    n_layers = model.cfg.n_layers
    layers = []
    for i in range(n_layers):
        resid = cache[f"blocks.{i}.hook_resid_post"].squeeze(0)  # (seq, d_model)
        resid_normed = model.ln_final(resid)            # apply final layernorm
        logits = model.unembed(resid_normed)            # (seq, vocab)
        layers.append(logits)
    return torch.stack(layers)  # (n_layers, seq, vocab)

all_logits = logit_lens(cache, model)  # (n_layers, seq, vocab)
probs = torch.softmax(all_logits, dim=-1)
all_logits = logit_lens(cache, model)  # (n_layers, seq, vocab) — remove batch dim
probs = torch.softmax(all_logits, dim=-1)

In [ ]:
top_tokens = all_logits.argmax(dim=-1).squeeze(1).cpu()  # (n_layers, seq)
top_token_strs = [[model.to_single_str_token(int(top_tokens[l, s]))
                   for s in range(len(tokens))]
                  for l in range(n_layers)]

print(all_logits.shape)
print(top_tokens.shape)
print(len(tokens))

fig1 = px.imshow(
    [[top_tokens[l, s].item() for s in range(len(tokens))] for l in range(n_layers)],
    labels=dict(x="Token Position", y="Layer", color="Token ID"),
    x=tokens,
    y=[f"L{i}" for i in range(n_layers)],
    title="Logit Lens — Top Predicted Token ID per Layer",
    color_continuous_scale="Viridis",
    text_auto=False,
)
# overlay the actual token strings
for l in range(n_layers):
    for s in range(len(tokens)):
        fig1.add_annotation(x=s, y=l, text=top_token_strs[l][s],
                            showarrow=False, font=dict(color="white", size=9))
fig1.show()

In [ ]:
for l in range(n_layers):
    for s in range(len(tokens)):
        fig1.add_annotation(x=s, y=l, text=top_token_strs[l][s],
                            showarrow=False, font=dict(color="white", size=9))
fig1.show()

In [ ]:
out = widgets.Output()
text_input = widgets.Text(
    value="The sky is blue and the grass is green.",
    description="Prompt:",
    layout=widgets.Layout(width="600px")
)

In [ ]:
def run_logit_lens(prompt):
    tokens = model.to_str_tokens(prompt)
    _, cache = model.run_with_cache(prompt)
    n_layers = model.cfg.n_layers

    layers = []
    for i in range(n_layers):
        resid = cache[f"blocks.{i}.hook_resid_post"].squeeze(0)  # (seq, d_model)
        resid_normed = model.ln_final(resid)
        logits = model.unembed(resid_normed)
        layers.append(logits)
    
    all_logits = torch.stack(layers)  # (n_layers, seq, vocab)
    top_tokens = all_logits.argmax(dim=-1).cpu()
    top_token_strs = [[model.to_single_str_token(int(top_tokens[l, s]))
                       for s in range(len(tokens))]
                      for l in range(n_layers)]
    return tokens, top_token_strs, n_layers


In [ ]:
def update(_):
    with out:
        clear_output(wait=True)
        prompt = text_input.value
        if not prompt.strip():
            return
        tokens, top_token_strs, n_layers = run_logit_lens(prompt)
        
        fig = go.Figure(go.Heatmap(
            z=[[ord(top_token_strs[l][s][0]) if top_token_strs[l][s] else 0 for s in range(len(tokens))]
               for l in range(n_layers)],
            x=tokens,
            y=[f"L{i}" for i in range(n_layers)],
            text=top_token_strs,
            texttemplate="%{text}",
            colorscale="Viridis",
            showscale=False,
        ))
        fig.update_layout(
            title=f"Logit Lens — {prompt[:40]}{'...' if len(prompt) > 40 else ''}",
            xaxis_title="Input Token",
            yaxis_title="Layer",
            height=400,
        )
        fig.show()

In [ ]:
text_input.observe(update, names="value")
display(text_input, out)
update(None) 